# PEDS — todos os experimentos (curvas de aprendizado + tabelas)

Este notebook roda os **5 surrogates do paper** (Fourier 16/25, Fisher 16/25, Maxwell 10),
mostra a **curva de aprendizado baseline vs PEDS** de cada um e, no fim, reproduz as
**2 tabelas da página 21**.

A lógica pesada (solvers, modelos, treino, plots) está em `src/` — este notebook só orquestra,
exatamente como os notebooks originais em `demos/` faziam `include("../src/PEDS.jl")`.

In [ ]:
# Célula 1 — setup
%matplotlib inline
import os, sys, torch
sys.path.append(os.path.abspath('..'))      # enxergar a pasta src/ na raiz

from src.peds_experiments import (
    EXPERIMENTS, run_diffusion_experiment, plot_learning_curve, plot_replication_tables,
)
from src.maxwell_experiment import run_maxwell_experiment

DATA_ROOT = os.path.abspath('../data')
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', DEVICE, '| experimentos de difusão:', list(EXPERIMENTS))

results = {}   # acumula os resultados dos 5 experimentos

## Experimentos 1–4: difusão (Fourier) e reação-difusão (Fisher)
Mesmo solver de difusão (validado), loss Huber. Cada célula mostra a curva inline.

In [ ]:
# Célula 2 — Fourier(16), Fourier(25), Fisher(16), Fisher(25)
for name in EXPERIMENTS:
    print(f'=== {name} ===')
    r = run_diffusion_experiment(name, DATA_ROOT, device=DEVICE)
    results[name] = r
    print(f"  PEDS={r['final_peds']:.3f} | NN-only={r['final_nn']:.3f} | "
          f"low-fi={r['lowfi']:.3f} | w={r['w']:.3f}")
    plot_learning_curve(r)        # renderiza a curva inline

## Experimento 5: Maxwell(10)
Física FDFD (mais lenta: um solve esparso por amostra), loss NLL complexa.
Use `--skip` mentalmente: se só quer os 4 de difusão, pule esta célula.

In [ ]:
# Célula 3 — Maxwell(10)
rm = run_maxwell_experiment(DATA_ROOT, device=DEVICE)   # pode levar alguns minutos
results['Maxwell(10)'] = rm
print(f"PEDS={rm['final_peds']:.3f} | NN-only={rm['final_nn']:.3f} | "
      f"low-fi={rm['lowfi']:.3f} | w={rm['w']:.3f}")
plot_learning_curve(rm)

## Tabelas de replicação (página 21 do paper)
Replicado **e** valor do paper lado a lado, para você ver se bateu.

In [ ]:
# Célula 4 — as 2 tabelas (Extended Data Table 2 e 3)
plot_replication_tables(results)